# Train the new model set on Colab — unified `component` + 6 condition specialists

**Architecture (per the eval re-design):**
- `pole` — already good (0.92); **not retrained here**, reuse your existing weights.
- `component` — ONE detector, 8 types (wire, h_insulator, v_insulator, crossarm_stright, top_crossarm,
  om_crossarm, vegetation, rust). Pole-crop. **`yolo26x` @1280** (data-rich main detector).
- `cond_*` (6) — per-family condition specialists, each on its component's crop. **`yolo26m` @1280**
  (small focused tasks ~400–1500 instances → medium model resists overfit; bump to 26l if val keeps rising).

**Pick the best GPU:** Runtime → Change runtime type → **A100**. Then Run all.
Datasets expected in Drive (built on the SSD): `component`, `cond_v_insulator`, `cond_h_insulator`,
`cond_straight_crossarm`, `cond_top_crossarm`, `cond_om_crossarm`, `cond_wire`.

In [ ]:
# @title 1) Reduce CUDA fragmentation (before torch) + GPU check
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")
assert torch.cuda.is_available(), "No CUDA GPU! Runtime -> Change runtime type -> GPU (A100)."
NAME=torch.cuda.get_device_name(0); VRAM=torch.cuda.get_device_properties(0).total_memory/1e9
print(f"GPU: {NAME} | VRAM {VRAM:.0f} GB | torch {torch.__version__} CUDA {torch.version.cuda}")

In [ ]:
# @title 2) Repo + deps
REPO="/content/repo"
!git clone https://github.com/arupa444/trainDronisight.git $REPO 2>/dev/null || (cd $REPO && git pull)
%cd $REPO
!pip -q install uv && uv pip install --system -e .
import torch; print("CUDA:", torch.cuda.is_available())

In [ ]:
# @title 3) Mount Drive
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# @title 4) Load the 7 subsets from Drive -> /content/data
import os, glob, zipfile, shutil
DRIVE_DATA = "/content/drive/MyDrive/dronisight"     # <-- adjust to where you put the DBs
SUBSETS = ["component", "cond_v_insulator", "cond_h_insulator", "cond_straight_crossarm",
           "cond_top_crossarm", "cond_om_crossarm", "cond_wire"]
DEST = "/content/data/yolo_train_db"; os.makedirs(DEST, exist_ok=True)
def locate(sub):
    for c in [f"{DRIVE_DATA}/yolo_train_db/{sub}", f"{DRIVE_DATA}/{sub}",
              f"{DRIVE_DATA}/yolo_train_db/{sub}.zip", f"{DRIVE_DATA}/{sub}.zip"]:
        if os.path.exists(c): return c
    h=glob.glob(f"{DRIVE_DATA}/**/{sub}*", recursive=True); return h[0] if h else None
for sub in SUBSETS:
    dst=f"{DEST}/{sub}"
    if os.path.isdir(dst) and glob.glob(f"{dst}/images/**/*.jpg", recursive=True): print("present:",sub); continue
    src=locate(sub); assert src, f"missing {sub} under {DRIVE_DATA}"
    if src.endswith(".zip"):
        with zipfile.ZipFile(src) as z: z.extractall(DEST)
        if not os.path.isdir(dst):
            inner=glob.glob(f"{DEST}/**/{sub}", recursive=True);  shutil.move(inner[0], dst) if inner else None
    else: shutil.copytree(src, dst, dirs_exist_ok=True)
    print("loaded:", sub)
os.environ["DRONISIGHT_DATA"]="/content/data"
import importlib, shared.config as C; importlib.reload(C)
for sub in SUBSETS:
    n=len(glob.glob(f"{C.YOLO_DB/sub}/images/train/clahe/*.jpg")); print(f"{sub}: train(clahe)={n}")
    for c in glob.glob(f"{C.YOLO_DB/sub}/**/*.cache", recursive=True): os.remove(c)

In [ ]:
# @title 5) Train — component (yolo26x) + 6 condition specialists (yolo26m). Saves to Drive per model.
from train_yolo.train_components import run
from notebooks.colab_utils import save_runs_to_drive

# (subset, model, imgsz, epochs). cond_* are small/focused -> yolo26m (anti-overfit).
PLAN = [
    ("component",              "yolo26x.pt", 1280, 150),
    ("cond_v_insulator",       "yolo26m.pt", 1280, 150),
    ("cond_h_insulator",       "yolo26m.pt", 1280, 150),
    ("cond_straight_crossarm", "yolo26m.pt", 1280, 150),
    ("cond_top_crossarm",      "yolo26m.pt", 1280, 150),
    ("cond_om_crossarm",       "yolo26m.pt", 1280, 150),
    ("cond_wire",              "yolo26m.pt", 1280, 150),
]
def pick_batch(model, vram):
    heavy = "26x" in model            # 26x@1280 is memory-bound; 26m fits much more
    table = [(78,6),(40,3),(22,2),(0,1)] if heavy else [(78,32),(40,16),(22,8),(0,4)]
    for thr,b in table:
        if vram>=thr: return b
    return 1
for sub, model, imgsz, epochs in PLAN:
    batch = pick_batch(model, VRAM)
    print("="*70, f"\nTRAIN {sub}  model={model} imgsz={imgsz} batch={batch} epochs={epochs}\n", "="*70)
    run(subset=sub, version="clahe", epochs=epochs, imgsz=imgsz, batch=batch, model=model)
    print("saved to Drive:", save_runs_to_drive())   # persist after EACH model (sessions are ephemeral)

In [ ]:
# @title 6) Per-model val mAP (best.pt)
import glob
from train_yolo.eval_yolo import evaluate
for sub in SUBSETS:
    w=sorted(glob.glob(f"runs/**/{sub}/**/weights/best.pt", recursive=True), key=__import__("os").path.getmtime)
    if w: print(evaluate(w[-1], sub, split="val", imgsz=1280), "\n")